# Heat Exchanger Network Design of the Ye and Grossman Four-Stream Problem
This notebook demonstrates the public heat exchanger network design workflow on the converted OpenHENS Ye and Grossman four-stream problem. It uses the problem-owned `design` accessor for direct use with `PinchProblem`.

## Imports

In [1]:
import pandas as pd
from OpenPinch import PinchProblem

## Ye and Grossman four-stream case
The stream and utility data come from the packaged `Four-stream-Yee-and-Grossmann-1990-1.json` conversion of the OpenHENS Ye and Grossman four-stream synthesis example. The packaged case keeps the original OpenHENS grid; this notebook narrows the grid to three approach temperatures, one derivative threshold, and one stage count so it can show the top three networks without sending Couenne through a broad teaching sweep.

In [2]:
case_name = "Four-stream-Yee-and-Grossmann-1990-1.json"

options_dict = {
    "HENS_APPROACH_TEMPERATURES": [6.0, 10.0, 14.0, 18.0],
    "HENS_DERIVATIVE_THRESHOLDS": [0.5],
    "HENS_STAGE_SELECTION": [5],
    "HENS_BEST_SOLUTIONS_TO_SAVE": 3,
    "HENS_MAX_PARALLEL": 10,
    "HENS_OUTPUT_FOLDER": "",
    "HENS_OUTPUT_FORMATS": [],
}

## Validate and prepare a problem
`PinchProblem` resolves the packaged sample-case name, validates the converted OpenHENS payload, and prepares the problem-owned `design` accessor used below.

In [3]:
problem = PinchProblem(
    source=case_name,
    project_name="Four-stream converted OpenHENS example",
)
problem.update_options(options_dict)
validated = problem.validate()

stream_frame = pd.DataFrame(
    problem.master_zone.all_streams.to_dict()
)

configuration_frame = pd.DataFrame(
    [
        {
            "source": case_name,
            "project": problem.project_name,
            "streams": len(validated.streams),
            "utilities": len(validated.utilities or []),
            "approach_temperatures": validated.options["HENS_APPROACH_TEMPERATURES"],
            "derivative_thresholds": validated.options["HENS_DERIVATIVE_THRESHOLDS"],
            "stage_selection": validated.options["HENS_STAGE_SELECTION"],
        }
    ]
)

display(stream_frame)
configuration_frame

,name,category,type,is_process_stream,t_supply,t_target,heat_flow,dt_cont,dt_cont_multiplier,htc,active
0,Raw Milk,hot_stream,Hot,True,376.85,96.85,2800.0,0.0,1.0,1.0,True
1,HT Flash,hot_stream,Hot,True,316.85,96.85,4400.0,0.0,1.0,1.0,True
2,Milk Concentrate,cold_stream,Cold,True,136.85,376.85,3600.0,0.0,1.0,1.0,True
3,CIP Water,cold_stream,Cold,True,76.85,226.85,1950.0,0.0,1.0,1.0,True
4,HPS,hot_utility,Hot,False,406.85,406.84,0.0,0.0,1.0,5.0,True
5,CW,cold_utility,Cold,False,26.85,46.85,0.0,0.0,1.0,1.0,True


,source,project,streams,utilities,approach_temperatures,derivative_thresholds,stage_selection
0,Four-stream-Yee-and-Grossmann-1990-1.json,Four-stream converted OpenHENS example,4,2,"[6.0, 10.0, 14.0, 18.0]",[0.5],[5]


## Run the design workflow
The public service entry point is owned by the problem: `problem.design.heat_exchanger_network_synthesis()`. By default this call invokes the local solver-backed synthesis executor. Heat exchanger network configuration belongs in the case options loaded above; runtime options are reserved for target execution state such as `state_id`. Running this cell requires the `openpinch[synthesis]` Python extra and external solver binaries; `idaes get-extensions` installs Couenne and Ipopt into the IDAES extension directory, which OpenPinch will discover automatically.

In [4]:
design = problem.design.heat_exchanger_network_synthesis()
top_ranked_network = design.ranked_networks[0]
network = design.network

pd.DataFrame(
    [
        {
            "run_id": design.run_id,
            "selected_rank": 1,
            "selected_method": design.method,
            "solver_name": design.solver_name,
            "solver_status": design.solver_status,
            "stage_count": design.stage_count,
            "ranked_network_count": len(design.ranked_networks),
            "exchanger_count": len(network.exchangers),
            "selected_network_is_rank_1": top_ranked_network.network == network,
            "cached_on_problem": problem.results.design == design,
        }
    ]
)

,run_id,selected_rank,selected_method,solver_name,solver_status,stage_count,ranked_network_count,exchanger_count,selected_network_is_rank_1,cached_on_problem
0,Four-stream-Yee-and-Grossmann-1990-1,1,network_evolution_method,ipopt-pyomo,1,3,2,6,True,True


## Inspect the workflow manifest
The result carries serializable workflow metadata alongside the selected network.

In [5]:
manifest = design.manifest

manifest_frame = pd.DataFrame(
    [
        {
            "run_id": manifest.run_id,
            "approach_temperatures": list(manifest.approach_temperatures),
            "derivative_thresholds": list(manifest.derivative_thresholds),
            "stage_selection": list(manifest.stage_selection),
            "task_ids": len(manifest.task_ids),
            "export_formats": list(manifest.export_formats),
        }
    ]
)

ranked_network_frame = pd.DataFrame(
    [
        {
            "method": outcome.task.method,
            "status": outcome.status,
            "approach_temperature_K": outcome.task.approach_temperature,
            "derivative_threshold": outcome.task.derivative_threshold,
            "stage_count": outcome.task.stage_count,
            "objective_value": outcome.objective_value,
        }
        for outcome in design.ranked_networks
    ]
)

display(manifest_frame)
ranked_network_frame

,run_id,approach_temperatures,derivative_thresholds,stage_selection,task_ids,export_formats
0,Four-stream-Yee-and-Grossmann-1990-1,"[6.0, 10.0, 14.0, 18.0]",[0.5],[5],12,[]


,method,status,approach_temperature_K,derivative_threshold,stage_count,objective_value
0,network_evolution_method,success,18.0,0.5,3,154853.857786
1,network_evolution_method,success,6.0,0.5,3,154853.858932


## Top network grid designs
Successful network candidates are exposed as `design.ranked_networks` and ranked by objective value after duplicate exchanger-connection structures are removed. `design.network` is the top-ranked network by default, and `design.select_network(solution_rank=...)` changes the selected network. The selected `HeatExchangerNetwork` draws itself with `design.network.build_grid_diagram(...)`. The Plotly grid diagrams below show the top three unique candidates from the accepted method family. Hover over exchanger markers to inspect match duty, area, and stream pairing.


In [6]:
top_ranked_networks = design.get_n_best_networks(3)

ranked_network_summary = pd.DataFrame(
    [
        {
            "rank": rank,
            "method": outcome.task.method,
            "task_id": outcome.task.task_id,
            "approach_temperature_K": outcome.task.approach_temperature,
            "derivative_threshold": outcome.task.derivative_threshold,
            "stage_count": outcome.task.stage_count,
            "objective_value": outcome.objective_value,
            "recovery_duty_kW": outcome.network.total_duty(kind="recovery"),
            "hot_utility_duty_kW": outcome.network.total_duty(kind="hot_utility"),
            "cold_utility_duty_kW": outcome.network.total_duty(kind="cold_utility"),
        }
        for rank, outcome in enumerate(top_ranked_networks, start=1)
    ]
)
display(ranked_network_summary)

selected_network_rows = [
    {
        "selected_rank": 1,
        "task_id": design.task_id,
        "objective_value": design.ranked_networks[0].objective_value,
        "exchanger_count": len(design.network.exchangers),
    }
]
if len(top_ranked_networks) >= 2:
    rank_2_design = design.model_copy(deep=True).select_network(solution_rank=2)
    selected_network_rows.append(
        {
            "selected_rank": 2,
            "task_id": rank_2_design.task_id,
            "objective_value": rank_2_design.ranked_networks[1].objective_value,
            "exchanger_count": len(rank_2_design.network.exchangers),
        }
    )

pd.DataFrame(selected_network_rows)

,rank,method,task_id,approach_temperature_K,derivative_threshold,stage_count,objective_value,recovery_duty_kW,hot_utility_duty_kW,cold_utility_duty_kW
0,1,network_evolution_method,hens-task-160fee34be4269f3,18.0,0.5,3,154853.857786,5068.852516,481.147460,2131.147460
1,2,network_evolution_method,hens-task-626df69d744638d8,6.0,0.5,3,154853.858932,5068.852532,481.147468,2131.147468


,selected_rank,task_id,objective_value,exchanger_count
0,1,hens-task-160fee34be4269f3,154853.857786,6
1,2,hens-task-626df69d744638d8,154853.858932,6


In [7]:
grid_diagrams = []

for rank in range(1, min(3, len(top_ranked_networks)) + 1):
    design.select_network(solution_rank=rank)
    diagram = design.network.build_grid_diagram()
    diagram.fig.update_layout(title_text=f"Rank {rank} heat exchanger network grid")
    grid_diagrams.append(diagram)
    display(diagram.fig)

design.select_network(solution_rank=1)
network = design.network

The returned diagram object wraps a Plotly figure. `diagram.save("network.png")` writes a static image through Kaleido, and `diagram.save("network.html")` writes an interactive Plotly document.


## Inspect the selected network
The selected `HeatExchangerNetwork` exposes ordered exchanger records plus stable labelled accessors for totals and stream-to-stream matches.

In [8]:
exchanger_frame = pd.DataFrame(
    [
        {
            "id": exchanger.exchanger_id,
            "kind": exchanger.kind.value,
            "source": exchanger.source_stream,
            "sink": exchanger.sink_stream,
            "stage": exchanger.stage,
            "duty_kW": exchanger.duty,
            "area": exchanger.area,
            "active": exchanger.active,
            "match_allowed": exchanger.match_allowed,
            "source_outlet_K": exchanger.source_outlet_temperature,
            "sink_outlet_K": exchanger.sink_outlet_temperature,
        }
        for exchanger in network.exchangers
    ]
)

exchanger_frame

,id,kind,source,sink,stage,duty_kW,area,active,match_allowed,source_outlet_K,sink_outlet_K
0,recovery:Process A.Raw Milk->Process A.Milk Co...,recovery,Process A.Raw Milk,Process A.Milk Concentrate,1.0,690.027116,75.752080,True,True,580.997288,617.923503
1,recovery:Process A.Raw Milk->Process A.CIP Wat...,recovery,Process A.Raw Milk,Process A.CIP Water,3.0,1950.000000,70.284246,True,True,385.997286,500.000000
2,recovery:Process A.HT Flash->Process A.Milk Co...,recovery,Process A.HT Flash,Process A.Milk Concentrate,3.0,2428.825400,141.038093,True,True,468.558730,571.921693
3,hot-utility:Hot Utility.HPS->Process A.Milk Co...,hot_utility,Hot Utility.HPS,Process A.Milk Concentrate,NaN,481.147460,13.089061,True,True,680.000000,650.000000
4,cold-utility:Process A.Raw Milk->Cold Utility.CW,cold_utility,Process A.Raw Milk,Cold Utility.CW,NaN,159.972859,4.706537,True,True,370.000000,320.000000
5,cold-utility:Process A.HT Flash->Cold Utility.CW,cold_utility,Process A.HT Flash,Cold Utility.CW,NaN,1971.174600,37.762310,True,True,370.000000,320.000000


In [9]:
first_recovery = next(
    exchanger
    for exchanger in network.exchangers
    if exchanger.kind == "recovery"
)
hot_utility_name = next(
    exchanger.source_stream
    for exchanger in network.exchangers
    if exchanger.kind == "hot_utility"
)
cold_utility_name = next(
    exchanger.sink_stream
    for exchanger in network.exchangers
    if exchanger.kind == "cold_utility"
)

pd.DataFrame(
    [
        {
            "metric": "total recovery duty",
            "value": problem.design.network.total_heat_recovery,
        },
        {
            "metric": "total hot utility duty",
            "value": problem.design.network.total_hot_utility,
        },
        {
            "metric": "total cold utility duty",
            "value": problem.design.network.total_cold_utility,
        },
        {
            "metric": f"{hot_utility_name} duty",
            "value": problem.design.network.utility(hot_utility_name),
        },
        {
            "metric": f"{cold_utility_name} duty",
            "value": problem.design.network.utility(cold_utility_name),
        },
        {
            "metric": "first recovery match duty",
            "value": network.labelled_value(
                "recovery_duty",
                source_stream=first_recovery.source_stream,
                sink_stream=first_recovery.sink_stream,
                stage=first_recovery.stage,
            ),
        },
    ]
)

,metric,value
0,total recovery duty,5068.852516
1,total hot utility duty,481.147460
2,total cold utility duty,2131.147460
3,Hot Utility.HPS duty,481.147460
4,Cold Utility.CW duty,2131.147460
5,first recovery match duty,690.027116
